In [ ]:
import sys
import time

import arviz as az
import numpy as np
from regina import Triangulation2
from tqdm import tqdm

# sys.path.append("./src/") 
from functions2d import choosemove
from functions2d_fast_accept_all import choosemove as choosemove_fast

In [6]:
alph = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789+-"
conv = {alph[i]: i for i in range(len(alph))}

In [9]:
gamma = 1 / 5
its = 100_000

isos_slow = ["cPbbde"] # torus (g=1) of 2 triangles
t = Triangulation2.fromIsoSig(isos_slow[0])  
f = t.fVector()
sizes_slow = [f[2]]   

start = time.perf_counter()
for c in tqdm(range(20)):
    for i in range(its):
        res = choosemove(isos_slow[-1], f, gamma)
        assert res
        iso, f = res
        isos_slow.append(iso)
        sizes_slow.append(f[2])
slow_time = time.perf_counter() - start


100%|██████████| 20/20 [06:51<00:00, 20.56s/it]


In [10]:
gamma = 1 / 5
its = 100_000
isos = ["cPbbde"]

f = None
aut = None
sizes_fast = [Triangulation2.fromIsoSig(isos[0]).size()] 

start = time.perf_counter()
for c in tqdm(range(20)):
    for i in range(its):
        iso, f, aut = choosemove_fast(isos[-1], gamma, f=f, aut=aut)
        isos.append(iso)
        sizes_fast.append(f[2])  
fast_time = time.perf_counter() - start

100%|██████████| 20/20 [01:11<00:00,  3.58s/it]


In [11]:
slow_n = np.array(sizes_slow)
fast_n = np.array(sizes_fast)

# ESS of the number of triangles, per step and per second of wall time.
slow_ess_total = az.ess(slow_n[None, :])
fast_ess_total = az.ess(fast_n[None, :])

print(f"original:   ESS/step {slow_ess_total / len(sizes_slow):.4f}, ESS/s {slow_ess_total / slow_time:.1f}, {slow_time:.0f}s")
print(f"accept-all: ESS/step {fast_ess_total / len(sizes_fast):.4f}, ESS/s {fast_ess_total / fast_time:.1f}, {fast_time:.0f}s")
print(f"speedup in ESS/s: {(fast_ess_total / fast_time) / (slow_ess_total / slow_time):.1f}x")

original:   ESS/step 0.0030, ESS/s 14.7, 411s
accept-all: ESS/step 0.0034, ESS/s 93.9, 72s
speedup in ESS/s: 6.4x
